# 02 · Análisis de la carga de cuidados

## Contexto

El trabajo doméstico y de cuidados no remunerado (TNR) es una de las formas más persistentes de desigualdad de género: es trabajo real, con valor económico demostrable, pero no se contabiliza en el PIB ni determina ingresos ni pensiones. En Chile, la Encuesta Nacional de Uso del Tiempo (ENUT) lo mide en 2015 y 2023.

Este notebook responde tres preguntas:
1. ¿La brecha se redujo entre 2015 y 2023?
2. ¿La carga recae más sobre las mujeres más pobres?
3. ¿Qué pasa con la carga global cuando una mujer trabaja de forma remunerada?

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from simel_client import SIMELClient

sns.set_theme(style='whitegrid', font_scale=1.05)
ROJO, AZUL = '#c0392b', '#2e86c1'

client = SIMELClient()
def cargar(ds):
    p = f'data/{ds}.csv'
    return pd.read_csv(p) if os.path.exists(p) else client.get(ds)

tnr_sexo  = cargar('DF_HRS_TNR_SEXO')
tnr_edad  = cargar('DF_HRS_TNR_SEXO_EDAD')
carga_tot = cargar('DF_HRS_TRAB_TOT_SEXO')
fdt       = cargar('DF_FDT_SEXO')
print('Datos cargados.')
print(tnr_sexo)

## 1. ¿Cuánto cambió la brecha entre 2015 y 2023?

In [ ]:
hm = tnr_sexo[tnr_sexo['SEXO'].isin(['M','F'])].copy()
anios = sorted(hm['AÑO'].unique())
x, w = np.arange(len(anios)), 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izq: barras
ax = axes[0]
for i, (sexo, color, label) in enumerate([('F',ROJO,'Mujeres'),('M',AZUL,'Hombres')]):
    vals = hm[hm['SEXO']==sexo].set_index('AÑO')['OBS_VALUE']
    b = ax.bar(x+i*w, [vals.get(a, np.nan) for a in anios],
               w, color=color, label=label, edgecolor='white', alpha=0.9)
    ax.bar_label(b, fmt='%.2fh', padding=3, fontsize=9)
ax.set_xticks(x+w/2); ax.set_xticklabels(anios)
ax.set_ylabel('Horas diarias'); ax.legend()
ax.set_title('Trabajo no remunerado por sexo', fontweight='bold')
sns.despine(ax=ax)

# Panel der: ratio
ax2 = axes[1]
ratios, diffs = [], []
for a in anios:
    f = hm[(hm['SEXO']=='F')&(hm['AÑO']==a)]['OBS_VALUE'].values
    m = hm[(hm['SEXO']=='M')&(hm['AÑO']==a)]['OBS_VALUE'].values
    if len(f) and len(m) and m[0]>0:
        ratios.append(f[0]/m[0]); diffs.append(f[0]-m[0])
    else:
        ratios.append(np.nan); diffs.append(np.nan)

ax2.bar([str(a) for a in anios], ratios, color=ROJO, edgecolor='white', width=0.5, alpha=0.85)
ax2.axhline(1, color='gray', linestyle='--', linewidth=1)
for i, (a, r, d) in enumerate(zip(anios, ratios, diffs)):
    if not np.isnan(r):
        ax2.text(i, r+0.02, f'×{r:.2f}\n({d:+.2f}h)', ha='center', fontsize=10, fontweight='bold')
ax2.set_ylabel('Ratio Mujeres / Hombres')
ax2.set_title('Brecha relativa', fontweight='bold')
sns.despine(ax=ax2)

plt.suptitle('Trabajo doméstico no remunerado — Chile 2015 vs 2023', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/brecha_cuidados.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretación

La brecha en horas de trabajo no remunerado **se redujo entre 2015 y 2023**, pero sigue siendo sustancial: las mujeres dedican aproximadamente el **doble** de tiempo que los hombres al trabajo doméstico y de cuidados.

La reducción se explica principalmente por una **caída en las horas de las mujeres**, no por un aumento en los hombres. Esto sugiere que el cambio responde más a mayor participación laboral femenina que a una redistribución real de las tareas del hogar.

> **Implicancia de política:** Las políticas de corresponsabilidad (salas cuna, permisos parentales compartidos) son más efectivas que las campañas de concientización para reducir esta brecha.

## 2. Ciclo de vida: ¿a qué edad es mayor la carga?

In [ ]:
edad_col = [c for c in tnr_edad.columns if 'EDAD' in c.upper()]
edad_col = edad_col[0] if edad_col else tnr_edad.columns[4]
e_max = tnr_edad['AÑO'].max()

fig, ax = plt.subplots(figsize=(12, 5))
for sexo, color, label in [('F',ROJO,'Mujeres'),('M',AZUL,'Hombres')]:
    sub = tnr_edad[(tnr_edad['SEXO']==sexo)&(tnr_edad['AÑO']==e_max)&(tnr_edad[edad_col]!='_T')].sort_values(edad_col)
    if len(sub):
        ax.plot(range(len(sub)), sub['OBS_VALUE'], 'o-', color=color, label=label, linewidth=2, markersize=7)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(sub[edad_col].values, rotation=30, ha='right')

ax.fill_between(range(len(sub)), 0, 1, alpha=0.05, color=ROJO, transform=ax.get_xaxis_transform())
ax.set_ylabel('Horas diarias'); ax.legend(); ax.set_title(f'Carga de cuidados por tramo etario — {e_max}', fontweight='bold')
sns.despine(); plt.tight_layout()
plt.savefig('outputs/figures/cuidados_por_edad.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretación

La distribución etaria muestra que la carga sobre las mujeres es **persistente en todos los tramos de edad**, sin excepción. El tramo 30–44 años — etapa de mayor productividad laboral — muestra la brecha más amplia.

> **Dato clave:** Las horas de trabajo no remunerado en ese tramo implican menor densidad de cotizaciones previsionales, lo que se traduce décadas después en **pensiones más bajas** para las mujeres.

## 3. Síntesis: el costo oculto en horas anuales

In [ ]:
hm_23 = tnr_sexo[tnr_sexo['SEXO'].isin(['M','F']) & (tnr_sexo['AÑO']==tnr_sexo['AÑO'].max())].set_index('SEXO')['OBS_VALUE']
if 'F' in hm_23.index and 'M' in hm_23.index:
    brecha_d = hm_23['F'] - hm_23['M']
    brecha_a = brecha_d * 365
    semanas  = brecha_a / 40
    valor_h  = 500000 / (45*4.33)
    costo_a  = brecha_a * valor_h
    print('=== Costo oculto de la brecha de cuidados ===')
    print(f'Brecha diaria:              {brecha_d:.2f} horas/día')
    print(f'Brecha anual:               {brecha_a:.0f} horas/año')
    print(f'Equivalente laboral:        {semanas:.1f} semanas de 40h')
    print(f'Valorizado a salario mínimo: ${costo_a:,.0f} CLP/año por mujer')